# 05 - Fourier sous Heston

Objectif: calculer le prix initial et le delta Fourier Heston, puis produire un P&L test.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np

from fourier.heston import compute_X0, compute_delta_asian_vectorized, compute_delta_vectorized
from models import compute_payoff_asian, compute_payoff_europeen, simulate_heston


In [ ]:
T, V0, K = 0.5, 0.0625, 1.0
kappa, theta, eta, rho = 4, V0, 0.7, -0.7
J, I = 40, 2_000

S, V, dS, sigma, states = simulate_heston(1, V0, kappa, theta, eta, rho, T, J, I, seed=5)
X0 = compute_X0(kappa=kappa, eta=eta, rho=rho, T=T, V0=V0, n_laguerre=64)
print('X0 Fourier Heston:', X0)


In [ ]:
delta = np.zeros((I, J))
for j in range(J):
    delta[:, j] = compute_delta_vectorized(j*T/J, S[:, j], V[:, j], kappa, theta, eta, rho, T, V0=V0, K=K, n_laguerre=64)

pnl = X0 + np.sum(delta * dS, axis=1) - compute_payoff_europeen(S, K)
print('MSE european:', np.mean(pnl**2), 'Mean PnL:', np.mean(pnl))


In [ ]:
bar_S = np.ones_like(S[:, 0])
delta_asian = np.zeros((I, J))
for j in range(J):
    if j > 0:
        bar_S = np.exp(np.mean(np.log(S[:, 1:j+1]), axis=1))
    delta_asian[:, j] = compute_delta_asian_vectorized(j*T/J, S[:, j], V[:, j], bar_S, kappa, theta, eta, rho, T, V0=V0, K=K, n_laguerre=32)

pnl_asian = np.sum(delta_asian * dS, axis=1) - compute_payoff_asian(S, K)
print('Asian hedge PnL without X0 term:', np.mean(pnl_asian), np.mean(pnl_asian**2))
